# Oppitunti 13 - Agentin muisti


## Asetus

Tässä muistikirjassa esitellään, miten rakennetaan matkavarausagentti, jolla on **pysyvä muisti**, käyttämällä **Microsoft Agent Frameworkia** (MAF).

Opit, miten eri tyyppiset agentin muistityypit — työmuisti, lyhytaikainen muisti ja pitkäaikainen muisti — vaikuttavat siihen, miten agentti säilyttää ja käyttää tietoa keskustelujen aikana.

**Esivaatimukset:**
- Azure AI Foundry -projekti, jossa on käytössä chat-malli (esim. `gpt-4o-mini`).
- Kirjautuneena Azure CLI:hin — suorita `az login` päätelaitteessasi.
- `AZURE_AI_PROJECT_ENDPOINT` — Azure AI Foundry -projektisi päätepiste.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — käytössä olevan mallin nimi.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Agentin muistityypit

AI-agentit voivat hyödyntää erilaisia muistityyppejä, joilla kullakin on oma tarkoituksensa:

### Työmuisti
Itsessään keskusteluketju – yhdessä istunnossa vaihdetut viestit. Agentti voi viitata aiempiin viesteihin samassa ketjussa säilyttääkseen johdonmukaisuuden. MAF:ssa tämä luodaan **`agent.create_session()`** -metodilla, joka palauttaa `AgentSession`-olion.

### Lyhytkestoinen muisti
Tieto, joka säilyy tehtävän tai istunnon ajan mutta ei tallennu pysyvästi. Esimerkiksi agentti voi kerätä faktoja monivaiheisen suunnittelukeskustelun aikana ja käyttää niitä lopullisen matkasuunnitelman laatimiseen.

### Pitkäkestoinen muisti
Mieltymykset ja faktat, jotka säilyvät **istuntojen välillä**. Palaavan käyttäjän ei pitäisi joutua toistamaan ruokavaliorajoituksiaan tai matkailutyyliään. Pitkäkestoinen muisti on tyypillisesti tuettu ulkoisella tallennustilalla – tietokanta, tiedosto tai vektori-indeksi – ja se välitetään agentille työkaluista.


## Työmuisti istuntojen kanssa

Yksinkertaisin muistin muoto on keskustelun istunto. Kun annat saman istunnon (luotu `agent.create_session()` -komennolla) peräkkäisiin `agent.run()`-kutsuihin, agentti näkee koko keskustelun historian ja voi muistaa aiemmat yksityiskohdat.

Luodaan matkailuagentti ja havainnollistetaan työmuistia.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agentti muisteli budjettia oikein, koska molemmat viestit ovat samassa istunnossa. Tämä on **työmuisti** — se on olemassa vain istunnon eliniän ajan.

### Mitä tapahtuu uudella ketjulla?

Jos luomme **uuden** istunnon, agentilla ei ole muistikuvaa aiemmasta keskustelusta:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Pitkän aikavälin muistimalli

Käyttäjäasetusten muistamiseksi **istuntojen yli** tarvitsemme pysyvän tallennustilan, joka sijaitsee keskusteluketjun ulkopuolella. Agentti käyttää tätä tallennustilaa **työkalujen** kautta — funktioita, joita se voi kutsua tietojen tallentamiseen ja noutamiseen.

Alla toteutamme yksinkertaisen muistissa olevan asetusten tallennustilan (tuotannossa tämä voisi olla tietokanta tai vektori-indeksi) ja teemme siitä työkaluja, joita agentti voi käyttää.

### Arkkitehtuuri
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Tapaus 1 — Ensimmäistä kertaa käyttäjä varaa vuosipäivämatkan

Sarah vierailee ensimmäistä kertaa. Asiakkaan tulisi tallentaa hänen mieltymyksensä työkalujen avulla ja käyttää niitä hotellien suositteluun.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Tapaus 2 — Sarah palaa viikkoja myöhemmin

Sarah aloittaa **täysin uuden keskustelun** (simuloiden uutta istuntoa). Työmuisti on tyhjä, mutta pitkäaikaisten mieltymysten tallennustilassa on edelleen hänen tietonsa. Agentin tulisi hakea ne ja käyttää niitä suositusten personointiin.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Yhteenveto

Tässä oppitunnissa opit kolme agentin muistityyppiä ja kuinka toteuttaa ne Microsoft Agent Frameworkilla:

| Muistityyppi | MAF-mekanismi | Kesto |
|---|---|---|
| **Työmuisti** | `agent.create_session()` | Yksi keskustelu |
| **Lyhytaikainen** | Kertynyt konteksti säikeessä | Yksi tehtävä / istunto |
| **Pitkäaikainen** | Ulkoinen tallennus, johon pääsee käsiksi `@tool`-funktioilla | Istuntojen yli |

### Tärkeimmät huomiot
1. **`agent.create_session()`** tarjoaa työmuistin — agentti näkee koko keskusteluhistorian istunnon aikana.
2. **Uudet istunnot menettävät kontekstin** — ilman pitkäaikaista muistia agentti ei muista aiempia keskusteluja.
3. **`@tool`-funktiot** yhdistävät aukon — ne antavat agentin tallentaa ja hakea tietoa pysyvästä tallennustilasta.
4. **Personalisointi paranee ajan myötä** — mitä enemmän mieltymyksiä tallennetaan, sitä parempia agentin suositukset ovat.

### Käytännön sovellukset
- **Asiakaspalvelu**: Muistaa asiakkaiden historian ja mieltymykset
- **Henkilökohtaiset avustajat**: Säilyttää kontekstin päivien tai viikkojen ajan
- **Terveydenhuolto**: Seuraa potilastietoja ja mieltymyksiä
- **Verkkokauppa**: Personoitu ostokokemus historian perusteella

### Seuraavat askeleet
- Korvaa muistissa oleva dict-tietorakenne tietokannalla tai vektorivarastolla (esim. Azure AI Search)
- Lisää muistin vanheneminen ajankriittisille tiedoille
- Rakenna moniagenttijärjestelmiä jaettuun muistiin perustuen
- Tutustu Cognee-muistikirjaan, jossa on tietokantapohjainen muisti


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:  
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Pyrimme tarkkuuteen, mutta huomioithan, että automaattikäännöksissä saattaa olla virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen omalla kielellä tulee pitää ensisijaisena ja luotettavana lähteenä. Tärkeissä asioissa suositellaan ammattilaisen tekemää käännöstä. Emme ole vastuussa tästä käännöksestä mahdollisesti aiheutuvista väärinymmärryksistä tai virhetulkinnasta.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
